# 03 — XAI Demo

Walk through SSM-GradCAM saliency map generation:
1. Load a trained checkpoint
2. Run inference on a sample image
3. Generate and overlay saliency heatmaps
4. Compare across stages and classes

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from medical_mamba.models import MedicalVMamba
from medical_mamba.data import get_val_transforms, DATASET_META
from medical_mamba.utils import load_checkpoint
from medical_mamba.xai import SSMGradCAM, overlay_heatmap

## 1. Load Model

In [ ]:
CHECKPOINT = "../outputs/best_model.pt"  # ← Update to your checkpoint
DS_NAME = "pathmnist"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

meta = DATASET_META[DS_NAME]
model = MedicalVMamba(task_classes={"default": meta["num_classes"]})
model, _, epoch, metrics = load_checkpoint(CHECKPOINT, model, device=DEVICE)
model = model.to(DEVICE).eval()
print(f"Loaded checkpoint from epoch {epoch}: {metrics}")

## 2. Prepare a Sample Image

In [ ]:
IMG_PATH = "../data/pathmnist_images/train/0/000000.jpg"  # ← Update path

transform = get_val_transforms(DS_NAME)
pil_img = Image.open(IMG_PATH).convert("RGB")
img_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

plt.imshow(pil_img)
plt.title("Original Image")
plt.axis("off")
plt.show()

## 3. Generate Saliency Maps

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
fig.suptitle("SSM-GradCAM Across Backbone Stages", fontsize=14)

for stage in range(4):
    cam = SSMGradCAM(model, target_stage=stage)
    heatmap = cam(img_tensor)
    hm_np = heatmap.cpu().numpy()
    img_np = np.array(pil_img.resize((224, 224)))
    overlay = overlay_heatmap(img_np, hm_np)
    axes[stage].imshow(overlay)
    axes[stage].set_title(f"Stage {stage}")
    axes[stage].axis("off")

plt.tight_layout()
plt.show()

## 4. Compare Across Classes

In [ ]:
cam = SSMGradCAM(model, target_stage=3)
n_classes = meta["num_classes"]

fig, axes = plt.subplots(1, min(n_classes, 5), figsize=(20, 4))
fig.suptitle("Saliency for Different Target Classes", fontsize=14)

for i, ax in enumerate(axes):
    if i >= n_classes:
        ax.axis("off")
        continue
    hm = cam(img_tensor, class_idx=i).cpu().numpy()
    img_np = np.array(pil_img.resize((224, 224)))
    overlay = overlay_heatmap(img_np, hm)
    ax.imshow(overlay)
    ax.set_title(f"Class {i}")
    ax.axis("off")

plt.tight_layout()
plt.show()